# Clustering de estaciones EcoBicis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasetting import cargar_datos, matriz_od, destinos_por_estacion, uso_por_estacion, aplicar_estilo

aplicar_estilo()

## Datos
Viajes válidos (1 min – 3 h, desde el 1 de septiembre) ya limpios, y catálogo de estaciones indexado por `num_cicloe`.

In [2]:
viajes, estaciones = cargar_datos()
print(f"{len(viajes):,} viajes, {len(estaciones):,} estaciones")
viajes.head()

1,498,099 viajes, 677 estaciones


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Ciclo_Estacion_Arribo,Fecha_Hora_Retiro,Fecha_Hora_Arribo,Duracion_min,Fecha,Hora,Dia_Semana,Fin_de_Semana,Viaje_Circular,Rango_Edad,Alcaldia_Retiro,Alcaldia_Arribo,Distancia_km,Velocidad_kmh
0,M,41,7483787,7,261,2026-09-01 00:00:28,2026-09-01 00:03:08,2.666667,2026-09-01,0,1,False,False,35-44,Cuauhtemoc,Cuauhtemoc,0.286911,6.455503
1,M,51,5717525,632,641,2026-09-01 00:01:38,2026-09-01 00:04:55,3.283333,2026-09-01,0,1,False,False,45-54,Benito Juarez,Benito Juarez,0.310796,5.679516
2,M,26,6196315,340,323,2026-09-01 00:02:24,2026-09-01 00:06:11,3.783333,2026-09-01,0,1,False,False,25-34,Benito Juarez,Benito Juarez,0.631568,10.016059
3,O,31,4826665,359,357,2026-09-01 00:03:46,2026-09-01 00:06:41,2.916667,2026-09-01,0,1,False,False,25-34,Benito Juarez,Benito Juarez,0.635490,13.072927
4,M,39,6283766,577,280,2026-09-01 00:00:20,2026-09-01 00:06:50,6.500000,2026-09-01,0,1,False,False,35-44,Cuauhtemoc,Benito Juarez,1.175099,10.847072


## Matriz origen-destino
Filas = estación de retiro, columnas = estación de arribo. `matriz_od.loc[27, 1]` = viajes de la 27 a la 1.

In [3]:
od = matriz_od(viajes)
od

Ciclo_Estacion_Arribo,1,2,3,4,5,6,7,8,9,10,...,702,703,704,705,706,707,708,709,710,711
Ciclo_Estacion_Retiro,,,,,,,,,,,,,,,,,,,,,
1,146,56,18,114,97,49,51,18,116,23,...,0,0,0,0,0,0,0,0,7,1
2,47,63,3,21,10,38,46,35,37,4,...,0,0,0,0,0,1,0,0,2,3
3,30,4,50,17,16,7,31,12,11,24,...,0,0,0,0,0,0,0,1,4,3
4,127,15,5,95,31,42,44,1,13,5,...,0,0,0,0,0,0,0,0,6,1
5,86,9,10,33,81,16,92,8,44,6,...,0,0,0,0,0,0,0,0,2,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,0,0,0,0,0,0,1,0,0,0,...,2,7,2,25,5,18,40,12,0,0
708,0,0,0,0,0,0,4,0,0,0,...,10,16,2,13,13,58,27,22,0,0
709,0,0,0,0,0,0,0,0,0,1,...,21,15,14,38,48,9,8,40,0,1


Versión simétrica (A→B + B→A) y sin viajes circulares, útil como medida de conexión entre estaciones para clustering.

In [4]:
od_sim = matriz_od(viajes, incluir_circulares=False, simetrica=True)
od_sim

Ciclo_Estacion_Arribo,1,2,3,4,5,6,7,8,9,10,...,702,703,704,705,706,707,708,709,710,711
Ciclo_Estacion_Retiro,,,,,,,,,,,,,,,,,,,,,
1,0,103,48,241,183,136,118,23,186,55,...,0,0,0,0,0,0,0,0,16,6
2,103,0,7,36,19,74,90,41,44,14,...,0,0,0,0,0,1,0,0,3,3
3,48,7,0,22,26,16,48,23,16,82,...,0,0,0,0,0,0,0,1,5,10
4,241,36,22,0,64,138,92,6,23,20,...,0,0,0,0,0,0,0,0,11,5
5,183,19,26,64,0,61,163,25,53,39,...,0,0,0,0,0,0,0,0,3,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,0,1,0,0,0,0,1,0,0,0,...,4,12,2,46,5,0,98,21,0,0
708,0,0,0,0,0,1,4,0,0,6,...,24,28,16,32,38,98,0,30,0,0
709,0,0,1,0,0,0,0,0,0,1,...,35,23,41,99,98,21,30,0,0,1


## Destinos por estación
Tabla de destinos de cada estación de origen, ordenada de mayor a menor número de viajes.

In [5]:
destinos = destinos_por_estacion(viajes, estaciones)

estacion = 27
print(f"Estación {estacion}: {estaciones.loc[estacion, 'Estacion']}")
destinos[estacion].head(15)

Estación 27: Av. Paseo de la Reforma y Havre


,Ciclo_Estacion_Arribo,Num_Viajes,Destino
0,271,227,Jesus Garcia y Carlos J. Meneses
1,27,204,Av. Paseo de la Reforma y Havre
2,43,182,Juarez y Doctor Mora
3,1,181,Rio Sena y Rio Balsas
4,260,145,Juarez y Balderas
5,29,143,Av. Paseo de la Reforma y Bucareli
6,52,132,Hidalgo y Trujano
7,17,115,Av. Paseo de la Reforma y Rio Tiber
8,14,109,Av. Paseo de la Reforma y Rio de la Plata
9,174,104,Joaquin Garcia Icazbalceta y Ignacio Manuel Al...
